<a href="https://colab.research.google.com/github/jlw44874/GameWise/blob/main/GameWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GameWise
My app helps budget-conscious PC gamers decide which steam game to buy by comparing games that fit their genre, budget, their playtime value, current deals, and computer compatibility.



In [1]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST4600/


Sub question 1: Which games are good and match the gamer’s preferred genre and budget?



In [7]:
def find_matching_games(df, genre, budget, min_rating=80):
    in_genre = df["genres"].str.contains(genre, case=False, na=False)
    in_budget = df["price"] <= budget
    is_good = df["pct_pos_total"] >= min_rating

    matches = df[in_genre & in_budget & is_good]
    columns = ["name", "genres", "price", "pct_pos_total", "num_reviews_total"]
    return matches[columns].sort_values("pct_pos_total", ascending=False)

Sub question 2:  Which matching games offer the most average playtime per dollar?

In [6]:
def rank_by_value(df, matches):
    games = df.loc[matches.index].copy()
    games = games[(games["price"] > 0) & (games["avg_playtime_hours"] > 0)]
    games["hours_per_dollar"] = (games["avg_playtime_hours"] / games["price"]).round(2)

    columns = ["name", "price", "avg_playtime_hours", "hours_per_dollar", "pct_pos_total"]
    return games[columns].sort_values("hours_per_dollar", ascending=False)